ALAÜLESANDE KAART — ROLL A
Andmete laadimine Supabasest


Supabasega ühendamiseks kasuta .env faili.

In [117]:
#Loo ühendus supabase kasutajaga
import os
from dotenv import load_dotenv

load_dotenv()

import os

import pandas as pd
from supabase import create_client

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_ANON_KEY")
)


print(os.getcwd())
#Andmete pärimine Supabase andmebaasist ja andmete laadimine Pandas DataFrame'i
def get_data(sales_table):
    data = []
    page_size = 1000
    page = 0

#Andmete pärimine lehekaupa, kuni kõik andmed on laaditud
    while True:
        response = (
            supabase
            .table(sales_table)
            .select('*')
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )
#Andmete lisamine DataFrame'i
        data.extend(response.data)

        print("Leht:", page, "ridu:", len(response.data))

        if len(response.data) < page_size:
            break
#Andmete laadimise lõpetamine, kui kõik andmed on laadi

#Andmete laadimise lõpetamine, kui kõik andmed on laaditud
        page += 1

    return pd.DataFrame(data)

#Andmete laadimine Supabase andmebaasist sales ja customers tabelitest

df_sales = get_data('sales')
df_customers = get_data('customers')

#Andmete ühendamine (merge) customer_id alusel, et lisada email ja first_name veerud df_sales DataFrame'i

df = pd.merge(df_sales, df_customers[['customer_id', 'email',
    'first_name']], on='customer_id', how='left')



c:\Users\kasutaja\Documents\DACA-Python\venv
Leht: 0 ridu: 1000
Leht: 1 ridu: 1000
Leht: 2 ridu: 1000
Leht: 3 ridu: 1000
Leht: 4 ridu: 1000
Leht: 5 ridu: 1000
Leht: 6 ridu: 1000
Leht: 7 ridu: 1000
Leht: 8 ridu: 1000
Leht: 9 ridu: 1000
Leht: 10 ridu: 118
Leht: 0 ridu: 1000
Leht: 1 ridu: 1000
Leht: 2 ridu: 1000
Leht: 3 ridu: 150


ALAÜLESANDE KAART — ROLL B
Andmete puhastamine

In [118]:
#Andmete raami kuvamine, et näha, kuidas andmed välja näevad pärast ühendamist
print(df.head())

   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method                 email first_name  
0          kaart                   NaN      Hille  
1      järelmaks    merl

In [119]:
import pandas as pd

# 1. Esialgne shape
print("Esialgne shape:", df.shape)

Esialgne shape: (10118, 14)


Duplukaatide leidmine ja eemaldamine

In [120]:
# 2. Kontrolli ja eemalda duplikaadid
print("Duplikaadid:", df.duplicated().sum())
df = df.drop_duplicates()

Duplikaadid: 0


Duplikaate kokku 0

NULL väärtuste kontrollimine ja NULL väärtuste eemaldamine.

In [121]:
# 3. Kontrolli NULL väärtusi
print("NULL-id enne puhastust:\n", df.isnull().sum())

df = df.dropna(
    subset=['customer_id', 'sale_date', 'total_price']
)
print("NULL-id puhastuse järel:\n", df.isnull().sum())

NULL-id enne puhastust:
 id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
email             1944
first_name         988
dtype: int64
NULL-id puhastuse järel:
 id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id          0
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3128
payment_method       0
email              956
first_name           0
dtype: int64


Kuupäevade ühtlustamine

In [122]:
# 4. Parsi kuupäevad
df['sale_date'] = pd.to_datetime(
    df['sale_date'],
    dayfirst=True,
    errors='coerce'
)

NULL väärtuste eemaldamine

In [123]:
# 5. Kontrolli ja eemalda negatiivsed / null total_price väärtused
print("Negatiivsed või null total_price väärtused:", (df['total_price'] <= 0).sum())

df = df[df['total_price'] > 0]

Negatiivsed või null total_price väärtused: 180


Puhastusraport

In [124]:
# 6. Puhastusraport
print("\nPUHASTUSRAPORT")
print("-" * 30)
print("Lõplik shape:", df.shape)
print("Unikaalseid kliente:", df['customer_id'].nunique())
print("Kuupäevavahemik:", df['sale_date'].min(), "kuni", df['sale_date'].max())


PUHASTUSRAPORT
------------------------------
Lõplik shape: (8950, 14)
Unikaalseid kliente: 2540
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2026-12-03 00:00:00


Enne andmete puhastamist 10118 rida ja 20 veergu.
Peale puhastamist 8950 rida ja 14 veergu. Unikaalseid kliente 2540.

ALAÜLESANDE KAART — ROLL C
ROLL: Analysis — RFM kliendisegmenteerimine

Arvuta iga kliendi kohta Recency, Frequency ja Monetary väärtused.
Määra RFM skoorid (1-5, kvintiilide alusel) ja loo kliendisegmendid
(VIP Champions, Loyal, Potential, At Risk, Lost).

In [135]:
#Viitekuupäevade loomine
today = pd.to_datetime('2025-02-28')

In [144]:
#RFM Koodiloomine
#Viite kuupäeva määramine
import pandas as pd
today = pd.to_datetime('2025-02-28')

#Reacency arvutamine
recency_df = df.groupby('customer_id')['sale_date'].max().reset_index()

#Freaquency arvutamine
df.groupby('customer_id')['sale_id'].count()

#Monatery arvutamine
df.groupby('customer_id')['total_price'].sum()

#Liida RFM ühte tabelisse
import pandas as pd

# 1. Recency
analysis_date = df['sale_date'].max()

# 1. Recency
recency_df = df.groupby('customer_id')['sale_date'].max().reset_index()
recency_df.columns = ['customer_id', 'last_purchase_date']

# Arvuta päevade arv viimase ostu kuupäevast kuni analüüsi kuupäevani
recency_df['recency_days'] = (
    analysis_date - recency_df['last_purchase_date']
).dt.days


# 2. Frequency
frequency_df = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']


# 3. Monetary
monetary_df = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary']


# 4. Liida R, F, M ühte tabelisse
rfm = pd.merge(recency_df, frequency_df, on='customer_id', how='left')
rfm = pd.merge(rfm, monetary_df, on='customer_id', how='left')


# 5. Lisa skoorid 1-5
# Recency on vastupidine: väiksem päevade arv = parem skoor
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

#Loo RFM skoor

print(rfm.head())

rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

rfm['RFM_Score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)

   customer_id last_purchase_date  recency_days  frequency  monetary R_score  \
0       2001.0         2024-08-06           190          1     73.90       4   
1       2005.0         2024-03-10           339          2    480.52       3   
2       2006.0         2023-09-11           520          1    327.06       2   
3       2008.0         2024-03-02           347          2    148.03       3   
4       2009.0         2024-04-02           316          1    347.24       3   

  F_score M_score  
0       1       1  
1       3       3  
2       1       3  
3       3       1  
4       1       3  


Skooride määramine

In [145]:
#Reacency, Frequency ja Monetary skooride arvutamine kvantiilide alusel
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

In [146]:
# Frequency skooride arvutamine kvantiilide alusel
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

In [147]:
#Monetary skooride arvutamine kvantiilide alusel
rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

Kuupäevade puhastus enne RFM arvutust

In [148]:
# Kontrolli sale_date veergu
print("sale_date tüüp enne:", df['sale_date'].dtype)
print(df['sale_date'].head())

# Teisenda sale_date kuupäevaks
df['sale_date'] = pd.to_datetime(
    df['sale_date'],
    dayfirst=True,
    errors='coerce'
)

# Kontrolli, mitu kuupäeva ei õnnestunud teisendada
print("Vigased kuupäevad:", df['sale_date'].isna().sum())

# Eemalda read, kus sale_date on puudu
df = df.dropna(subset=['sale_date']).copy()

# Jäta alles ainult vajalik kuupäevavahemik
df = df[
    (df['sale_date'] >= '2023-01-01') &
    (df['sale_date'] <= '2025-02-28')
].copy()

print("Pärast kuupäevade puhastust:", df.shape)
print("Kuupäevavahemik:", df['sale_date'].min(), "kuni", df['sale_date'].max())

sale_date tüüp enne: datetime64[us]
0   2023-10-01
2   2023-05-01
3   2023-02-01
8   2023-07-01
9   2023-08-01
Name: sale_date, dtype: datetime64[us]
Vigased kuupäevad: 0
Pärast kuupäevade puhastust: (3288, 14)
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2025-02-12 00:00:00


RFM skoori arvutamine

In [149]:
print(rfm.head())

rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

rfm['RFM_Score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)

   customer_id last_purchase_date  recency_days  frequency  monetary R_score  \
0       2001.0         2024-08-06           190          1     73.90       4   
1       2005.0         2024-03-10           339          2    480.52       3   
2       2006.0         2023-09-11           520          1    327.06       2   
3       2008.0         2024-03-02           347          2    148.03       3   
4       2009.0         2024-04-02           316          1    347.24       3   

  F_score M_score  RFM_Score  
0       1       1          6  
1       3       3          9  
2       1       3          6  
3       3       1          7  
4       1       3          7  


In [150]:

# Kontrolli sale_date veergu
print("sale_date tüüp enne:", df['sale_date'].dtype)
print(df['sale_date'].head())

# Teisenda sale_date kuupäevaks
df['sale_date'] = pd.to_datetime(
    df['sale_date'],
    dayfirst=True,
    errors='coerce'
)

# Kontrolli, mitu kuupäeva ei õnnestunud teisendada
print("Vigased kuupäevad:", df['sale_date'].isna().sum())

# Eemalda read, kus sale_date on puudu
df = df.dropna(subset=['sale_date']).copy()

# Jäta alles ainult vajalik kuupäevavahemik
df = df[
    (df['sale_date'] >= '2023-01-01') &
    (df['sale_date'] <= '2025-02-28')
].copy()

print("Pärast kuupäevade puhastust:", df.shape)
print("Kuupäevavahemik:", df['sale_date'].min(), "kuni", df['sale_date'].max())

sale_date tüüp enne: datetime64[us]
0   2023-10-01
2   2023-05-01
3   2023-02-01
8   2023-07-01
9   2023-08-01
Name: sale_date, dtype: datetime64[us]
Vigased kuupäevad: 0
Pärast kuupäevade puhastust: (3288, 14)
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2025-02-12 00:00:00


Segmenteerimine

In [151]:
def segment(score):

    if score >= 18:
        return 'VIP Champions'

    elif score >= 14:
        return 'Loyal Customers'

    elif score >= 10:
        return 'Regular Customers'

    elif score >= 7:
        return 'New Customers'

    elif score >= 5:
        return 'At Risk'

    else:
        return 'Lost'

Rakendamine

In [152]:
rfm['Segment'] = (
    rfm['RFM_Score']
    .apply(segment)
)

Segmentide kokkuvõte

In [153]:
segment_summary = (
    rfm['Segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = [
    'Segment',
    'Kliente'
]
print("\nSegmentide kokkuvõte:")
print(segment_summary)


Segmentide kokkuvõte:
             Segment  Kliente
0  Regular Customers      602
1      New Customers      527
2            At Risk      294
3    Loyal Customers      169
4               Lost      137


Osakaalu lisamine

In [154]:
segment_summary['Osakaal_%'] = round(
    segment_summary['Kliente']
    / segment_summary['Kliente'].sum()
    * 100,
    1
)

print(segment_summary)

             Segment  Kliente  Osakaal_%
0  Regular Customers      602       34.8
1      New Customers      527       30.5
2            At Risk      294       17.0
3    Loyal Customers      169        9.8
4               Lost      137        7.9


ALAÜLESANDE KAART — ROLL D
Visualiseerimine ja leiud

In [155]:
import plotly.express as px
import pandas as pd

rfm = rfm.rename(columns={'monetary': 'monetary_value'})

1. Diagramm segmentide jaotuse kohta

In [156]:
segment_summary = (
    rfm['Segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = ['Segment', 'Kliente']

segment_summary = segment_summary.sort_values(
    'Kliente',
    ascending=False
)

fig1 = px.bar(
    segment_summary,
    x='Segment',
    y='Kliente',
    text='Kliente',
    color='Kliente',
    color_continuous_scale='Greens',
    title='Klientide jaotus RFM segmentide järgi',
    labels={
        'Segment': 'RFM segment',
        'Kliente': 'Klientide arv'
    }
)

fig1.update_traces(textposition='outside')
fig1.update_layout(title_x=0.5, plot_bgcolor='white')

fig1.show()

2. Recency vs Monetery

In [157]:
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary_value',
    color='Segment',
    size='frequency',
    hover_data=['customer_id', 'RFM_Score'],
    title='Kliendid: viimasest ostust möödunud päevad ja kogukulutus',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary_value': 'Kogukulutus (€)',
        'frequency': 'Ostude arv',
        'Segment': 'Segment'
    }
)

fig2.update_layout(title_x=0.5, plot_bgcolor='white')

fig2.show()

3. TOP 10 klienti

In [159]:
import plotly.express as px

top_vip = (
    rfm
    .nlargest(10, 'monetary_value')
    .sort_values('monetary_value')
)

fig3 = px.bar(
    top_vip,
    x='monetary_value',
    y='customer_id',
    orientation='h',
    text='monetary_value',
    color='monetary_value',
    color_continuous_scale='Greens',
    title='Top 10 klienti kogukulutuse järgi',
    labels={
        'customer_id': 'Klient',
        'monetary_value': 'Kogukulutus (€)'
    }
)

fig3.update_traces(
    texttemplate='€%{x:,.0f}',
    textposition='outside',
    textfont=dict(size=18, color='black')
)

fig3.update_layout(
    title_x=0.5,
    title_font=dict(size=26),
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    xaxis_tickfont=dict(size=14),
    yaxis_tickfont=dict(size=14),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=600
)

fig3.show()

Kaalutletud RFM skoori arvutamine

In [160]:
import pandas as pd

# Kui rahaveeru nimi on "monetary", aga edaspidi tahame kasutada "monetary_value"
if 'monetary_value' not in rfm.columns and 'monetary' in rfm.columns:
    rfm = rfm.rename(columns={'monetary': 'monetary_value'})

# Kontrollime, et skoorid oleksid numbrid
rfm = rfm.dropna(
    subset=['R_score', 'F_score', 'M_score']
).copy()

rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

# Kaalutud RFM skoor
# Monetary on 2x kaaluga
rfm['Weighted_RFM_Score'] = (
    rfm['R_score']
    + rfm['F_score']
    + 2 * rfm['M_score']
)

print(rfm.head())

   customer_id last_purchase_date  recency_days  frequency  monetary_value  \
0       2001.0         2024-08-06           190          1           73.90   
1       2005.0         2024-03-10           339          2          480.52   
2       2006.0         2023-09-11           520          1          327.06   
3       2008.0         2024-03-02           347          2          148.03   
4       2009.0         2024-04-02           316          1          347.24   

   R_score  F_score  M_score  RFM_Score        Segment  Weighted_RFM_Score  
0        4        1        1          6        At Risk                   7  
1        3        3        3          9  New Customers                  12  
2        2        1        3          6        At Risk                   9  
3        3        3        1          7  New Customers                   8  
4        3        1        3          7  New Customers                  10  


Täpsem segmentimine

In [161]:
def weighted_segment(score):
    if score >= 17:
        return 'VIP Champions'
    elif score >= 14:
        return 'Loyal Customers'
    elif score >= 11:
        return 'Regular Customers'
    elif score >= 9:
        return 'New Customers'
    elif score >= 7:
        return 'At Risk'
    else:
        return 'Lost'


rfm['Weighted_Segment'] = rfm['Weighted_RFM_Score'].apply(weighted_segment)

print(rfm[['customer_id', 'R_score', 'F_score', 'M_score', 'Weighted_RFM_Score', 'Weighted_Segment']].head())

   customer_id  R_score  F_score  M_score  Weighted_RFM_Score  \
0       2001.0        4        1        1                   7   
1       2005.0        3        3        3                  12   
2       2006.0        2        1        3                   9   
3       2008.0        3        3        1                   8   
4       2009.0        3        1        3                  10   

    Weighted_Segment  
0            At Risk  
1  Regular Customers  
2      New Customers  
3            At Risk  
4      New Customers  


Segmentide kokkuvõte

In [162]:
weighted_segment_summary = (
    rfm['Weighted_Segment']
    .value_counts()
    .reset_index()
)

weighted_segment_summary.columns = ['Segment', 'Kliente']

weighted_segment_summary['Osakaal_%'] = round(
    weighted_segment_summary['Kliente']
    / weighted_segment_summary['Kliente'].sum()
    * 100,
    1
)

weighted_segment_summary = weighted_segment_summary.sort_values(
    'Kliente',
    ascending=False
)

print(weighted_segment_summary)

             Segment  Kliente  Osakaal_%
0  Regular Customers      356       20.6
1    Loyal Customers      352       20.4
2      VIP Champions      324       18.7
3      New Customers      251       14.5
4            At Risk      232       13.4
5               Lost      214       12.4


Käibe osakaal segmentide kaupa

In [163]:
segment_revenue_summary = (
    rfm.groupby('Weighted_Segment')
    .agg(
        Kliente=('customer_id', 'count'),
        Kaive=('monetary_value', 'sum')
    )
    .reset_index()
)

segment_revenue_summary['Klientide_osakaal_%'] = round(
    segment_revenue_summary['Kliente']
    / segment_revenue_summary['Kliente'].sum()
    * 100,
    1
)

segment_revenue_summary['Kaibe_osakaal_%'] = round(
    segment_revenue_summary['Kaive']
    / segment_revenue_summary['Kaive'].sum()
    * 100,
    1
)

segment_revenue_summary = segment_revenue_summary.sort_values(
    'Kaive',
    ascending=False
)

print(segment_revenue_summary)

    Weighted_Segment  Kliente      Kaive  Klientide_osakaal_%  Kaibe_osakaal_%
5      VIP Champions      324  458194.88                 18.7             46.4
2    Loyal Customers      352  238823.36                 20.4             24.2
4  Regular Customers      356  155905.93                 20.6             15.8
3      New Customers      251   68719.22                 14.5              7.0
0            At Risk      232   42475.76                 13.4              4.3
1               Lost      214   24322.51                 12.4              2.5


Joonise koostamine

In [166]:
import plotly.express as px

# Sorteerime joonise jaoks väiksemast suuremani,
# et suurim käive jääks horisontaalsel joonisel üles/alla loetavalt
segment_revenue_plot = segment_revenue_summary.sort_values(
    'Kaive',
    ascending=True
)

fig = px.bar(
    segment_revenue_plot,
    x='Kaive',
    y='Weighted_Segment',
    orientation='h',
    text='Kaive',
    color='Kaive',
    color_continuous_scale='Greens',
    title='Käive kaalutud RFM segmentide kaupa',
    labels={
        'Weighted_Segment': 'Kaalutud RFM segment',
        'Kaive': 'Käive (€)'
    }
)

fig.update_traces(
    texttemplate='€%{x:,.0f}',
    textposition='outside',
    textfont=dict(
        size=16,
        color='black'
    )
)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=26),
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    xaxis_tickfont=dict(size=14),
    yaxis_tickfont=dict(size=14),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=600,
    margin=dict(l=160, r=120, t=90, b=70),
    coloraxis_colorbar=dict(
        title='Käive (€)'
    )
)

fig.show()

Eksport csv failina

In [167]:
export_columns = [
    'customer_id',
    'last_purchase_date',
    'recency_days',
    'frequency',
    'monetary_value',
    'R_score',
    'F_score',
    'M_score',
    'RFM_Score',
    'Segment',
    'Weighted_RFM_Score',
    'Weighted_Segment'
]

rfm_export = rfm[export_columns].copy()

rfm_export.to_csv(
    'rfm_segments.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Fail salvestatud: rfm_segments.csv")
print("Eksporditud ridu:", len(rfm_export))

Fail salvestatud: rfm_segments.csv
Eksporditud ridu: 1729


Kontroll, kuhu fail on salvestatud

In [168]:
import os

print("Faili asukoht:")
print(os.getcwd())

Faili asukoht:
c:\Users\kasutaja\Documents\DACA-Python\venv
